# Detektor anomalii w ruchu sieciowym

Plik `train.csv.gz` (w tym samym folderze co notatnik) zawiera kilka dni **normalnego** ruchu ośmiu sieci małych
firm – przepływy sieciowe z sondy Tstat, bez etykiet. Kolumna `network` mówi, z której sieci pochodzi przepływ. Zbuduj detektor, który nauczy się tego ruchu i będzie oceniał
nowe przepływy na dwa sposoby:

- `score(df)` – **ryzyko**: liczba dla każdego przepływu, im wyżej, tym bardziej podejrzany,
- `predict(df)` – **decyzja**: 1 = alarm, 0 = brak alarmu.

Efektem pracy jest plik **`model.pkl`** – wytrenowany detektor zapisany na końcu notatnika.

Kolumny: `network`, `ts` (s, UTC), `duration` (s), `src_ip`, `src_port`, `dst_ip`, `dst_port`, `proto` (tcp/udp),
`state` (SF, S0, REJ, RSTO, RSTR, OTH), `bytes_out`, `bytes_in`, `pkts_out`, `pkts_in`,
`sni` (nazwa serwera z TLS), `fqdn` (nazwa z DNS/HTTP, jeśli znana).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import cloudpickle


## Dane

In [ ]:
train = pd.read_csv("train.csv.gz", keep_default_na=False)
print(f"{len(train):,} przepływów")
train.head()


## Cechy

In [ ]:
FAILED = {"S0", "REJ", "RSTO", "RSTR", "OTH"}


def flow_features(df):
    """Cechy pojedynczego przepływu."""
    out = pd.DataFrame(index=df.index)
    out["log_bytes_out"] = np.log1p(df.bytes_out)
    out["log_bytes_in"] = np.log1p(df.bytes_in)
    out["log_pkts_out"] = np.log1p(df.pkts_out)
    out["log_duration"] = np.log1p(df.duration)
    out["out_in_ratio"] = np.log1p(df.bytes_out) - np.log1p(df.bytes_in)
    out["failed"] = df.state.isin(FAILED).astype(float)
    out["udp"] = (df.proto == "udp").astype(float)
    out["dst_port_low"] = (df.dst_port < 1024).astype(float)
    hour = (df.ts % 86400) / 3600
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return out


def window_features(df, window=60):
    """Agregaty w oknach czasowych po źródle i po celu, dołączone do każdego przepływu (skala log1p)."""
    w = (df.ts // window).astype(np.int64)
    tmp = df.assign(_w=w, _failed=df.state.isin(FAILED).astype(float))
    g = tmp.groupby(["src_ip", "_w"])
    src = pd.DataFrame({"src_n_flows": g.size(), "src_n_dst": g.dst_ip.nunique(), "src_n_dport": g.dst_port.nunique(),
                        "src_failed_ratio": g._failed.mean(), "src_bytes_out": g.bytes_out.sum()})
    g = tmp.groupby(["dst_ip", "_w"])
    dst = pd.DataFrame({"dst_n_flows": g.size(), "dst_n_src": g.src_ip.nunique(), "dst_n_dport": g.dst_port.nunique(),
                        "dst_failed_ratio": g._failed.mean()})
    out = tmp[["src_ip", "dst_ip", "_w"]].join(src, on=["src_ip", "_w"]).join(dst, on=["dst_ip", "_w"])
    out = out.drop(columns=["src_ip", "dst_ip", "_w"])
    for c in ("src_n_flows", "src_n_dst", "src_n_dport", "src_bytes_out", "dst_n_flows", "dst_n_src", "dst_n_dport"):
        out[c] = np.log1p(out[c])
    return out


class Familiarity:
    """Jak często w treningu widzieliśmy: adres docelowy, parę (src, dst), usługę (dst:port)."""

    def fit(self, df):
        self.dst = df.dst_ip.value_counts().to_dict()
        self.pair = (df.src_ip + ">" + df.dst_ip).value_counts().to_dict()
        self.svc = (df.dst_ip + ":" + df.dst_port.astype(str)).value_counts().to_dict()
        return self

    def transform(self, df):
        out = pd.DataFrame(index=df.index)
        out["fam_dst"] = np.log1p(df.dst_ip.map(self.dst).fillna(0).to_numpy())
        out["fam_pair"] = np.log1p((df.src_ip + ">" + df.dst_ip).map(self.pair).fillna(0).to_numpy())
        out["fam_svc"] = np.log1p((df.dst_ip + ":" + df.dst_port.astype(str)).map(self.svc).fillna(0).to_numpy())
        return out


## Detektor

In [ ]:
class Detector:
    """Punkt wyjścia: Isolation Forest na cechach przepływu, agregatach okien i znajomości celu.

    fit(df)     – uczy się na normalnym ruchu
    score(df)   – ryzyko dla każdego wiersza df; im wyżej, tym bardziej podejrzany przepływ
    predict(df) – decyzja dla każdego wiersza df: 1 = alarm, 0 = brak alarmu
    """

    def __init__(self, n_estimators=200, max_samples=2048, sample=150_000, seed=0, quantile=0.999):
        self.model = IsolationForest(n_estimators=n_estimators, max_samples=max_samples,
                                     random_state=seed, n_jobs=-1)
        self.sample, self.seed, self.quantile = sample, seed, quantile

    def _X(self, df):
        return flow_features(df).join(window_features(df)).join(self.fam.transform(df)).to_numpy(np.float32)

    def fit(self, df):
        self.fam = Familiarity().fit(df)
        X = self._X(df)
        if len(X) > self.sample:
            X = X[np.random.default_rng(self.seed).choice(len(X), self.sample, replace=False)]
        self.model.fit(X)
        # decyzja: alarm, gdy ryzyko przekracza próg – tu najprostszy wybór: kwantyl ryzyka na normalnym ruchu
        self.threshold = float(np.quantile(self.score(df), self.quantile))
        return self

    def score(self, df):
        return -self.model.score_samples(self._X(df))

    def predict(self, df):
        return (self.score(df) > self.threshold).astype(int)


## Trening

In [ ]:
det = Detector().fit(train)


In [ ]:
# ile alarmów podniósłby detektor na normalnym ruchu i które przepływy są najbardziej nietypowe
alarmy = det.predict(train)
godzin = (train.ts.max() - train.ts.min()) / 3600
print(f"alarmy na danych treningowych: {alarmy.sum():,} (ok. {alarmy.sum() / godzin / train.network.nunique():.1f} na godzinę na sieć)")
train.assign(score=det.score(train), alarm=alarmy).nlargest(10, "score")[
    ["src_ip", "dst_ip", "dst_port", "proto", "state", "bytes_out", "bytes_in", "score", "alarm"]]


## Zapis modelu

In [ ]:
with open("model.pkl", "wb") as f:
    cloudpickle.dump(det, f)   # cloudpickle zapisuje obiekt razem z klasą Detector i funkcjami cech

# kontrola: model wczytany z pliku daje te same wyniki
with open("model.pkl", "rb") as f:
    loaded = cloudpickle.load(f)
assert np.allclose(loaded.score(train.head(1000)), det.score(train.head(1000)))
assert (loaded.predict(train.head(1000)) == det.predict(train.head(1000))).all()
print("zapisano model.pkl")


## Ewaluacja
Plik `test.csv.gz` z etykietami w tym samym folderze.

In [ ]:
def evaluate(model, path="test.csv.gz"):
    """Prosta ocena decyzji modelu na pliku z etykietami (kolumny label i phase).
    Incydent = faza ataku w jednej sieci."""
    test = pd.read_csv(path, keep_default_na=False)
    X = test.drop(columns=[c for c in ("label", "phase", "raw_label") if c in test.columns])
    alarm = np.asarray(model.predict(X)).astype(bool)
    y = test.label.to_numpy().astype(bool)

    sieci = test.network.nunique()
    godzin = max((test.ts.max() - test.ts.min()) / 3600, 1)
    falszywe = int((alarm & ~y).sum())
    fpr = falszywe / max((~y).sum(), 1)
    print(f"sieci: {sieci}, przepływy: {len(test):,} (atak: {y.sum():,}), alarmy: {alarm.sum():,}")
    print(f"fałszywe alarmy: {falszywe:,} ({falszywe / godzin / sieci:.1f} na godzinę na sieć, FPR {fpr:.4f})")
    print(f"czułość: {(alarm & y).sum() / max(y.sum(), 1):.3f}, precyzja: {(alarm & y).sum() / max(alarm.sum(), 1):.3f}")

    fazy = []
    for (siec, faza), g in test[y].groupby(["network", "phase"]):
        a = alarm[g.index]
        czulosc = a.mean()
        fazy.append({"sieć": siec, "faza": faza, "przepływów": len(g), "alarmów": int(a.sum()),
                     "czułość": round(czulosc, 3),
                     # faza uznana za wykrytą, gdy alarmów jest w niej wyraźnie więcej niż przypadkiem (≥ 3 × FPR)
                     "wykryta": bool(a.any() and czulosc >= 3 * fpr)})
    fazy = pd.DataFrame(fazy)
    print(f"wykryte fazy: {fazy.wykryta.sum()} z {len(fazy)}")
    print(fazy.groupby("faza").wykryta.agg(["sum", "count"]).rename(columns={"sum": "wykryte", "count": "w sieciach"}))
    return fazy


In [ ]:
evaluate(det)

## Proponowane rozszerzenia

Szablon powyżej to punkt wyjścia. Poniżej:

- `regularity_features` – przykład cechy liczonej na grupach przepływów, regularność komunikacji w parze (src, dst, dst_port)
- `burst_features` – ile razy źródło połączyło się z jedną usługą (dst_ip:dst_port) w oknie i jaki odsetek tych połączeń był krótki; entropia portów docelowych źródła w oknie. Wzorzec: window_features wyżej
- `RankScaler` – narzędzie do łączenia wyników kilku detektorów.

Nowe cechy dołącz w metodzie `_X` swojego detektora, np.
`flow_features(df).join(window_features(df)).join(self.fam.transform(df)).join(regularity_features(df))`.
Cechy liczone na grupach (jak `regularity_features`) korzystają z całej ramki przekazanej do `score()`/`predict()`.


In [ ]:
def regularity_features(df, min_flows=5):
    """Przykład: regularność komunikacji w parze (src, dst, dst_port).
    cv = odchylenie standardowe / średnia; im bliżej 0, tym bardziej regularnie (cecha = 1 / (1 + cv)).
    Liczone osobno w każdej dobie, żeby trening (wiele dni) i nowe dane (jeden dzień) były porównywalne."""
    raise NotImplementedError


def burst_features(df, window=60):
    """Do uzupełnienia: tempo i różnorodność w oknie czasowym.

    Pomysły: ile razy źródło połączyło się z jedną usługą (dst_ip:dst_port) w oknie i jaki odsetek tych połączeń
    był krótki; entropia portów docelowych źródła w oknie. Wzorzec: window_features wyżej."""
    raise NotImplementedError


In [ ]:
class RankScaler:
    """Zamienia surowy wynik detektora na kwantyl względem wyników na normalnym ruchu (0–1).
    Dzięki temu wyniki różnych detektorów mają wspólną skalę i można je łączyć (średnia, maksimum...).

    rs = RankScaler().fit(wyniki_na_treningu)
    rs.transform(wyniki_na_nowych_danych)"""

    def fit(self, s, keep=200_000):
        s = np.sort(np.asarray(s))
        self.ref = s[np.linspace(0, len(s) - 1, min(keep, len(s))).astype(int)]
        return self

    def transform(self, s):
        return np.searchsorted(self.ref, s, side="right") / len(self.ref)


In [ ]:
# podgląd: jak wygląda nowa cecha na danych testowych w podziale na fazy
test = pd.read_csv("test.csv.gz", keep_default_na=False)
regularity_features(test).groupby(test.phase.replace("", "(normalny ruch)")).mean().round(3)


Gdy Twój detektor wypada lepiej niż szablon, ustaw `det` na nowy model i uruchom ponownie komórkę
z sekcji **Zapis modelu**.